# Day 4 模块 3：参数 vs 超参数

调模型之前，先分清两样东西：

- **参数**：模型在 `fit` 时**自己学出来的**
- **超参数**：你在训练**之前**就要定的旋钮


In [ ]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path('day4_cafe_sales.csv'),
    Path('day4') / 'day4_cafe_sales.csv',
    Path('教学课程') / 'day4' / 'day4_cafe_sales.csv',
]
for path in candidate_paths:
    if path.exists():
        csv_path = path
        break
else:
    raise FileNotFoundError('未找到 day4_cafe_sales.csv')

df = pd.read_csv(csv_path)
print(csv_path.resolve())
print('shape:', df.shape)
df.head()


In [ ]:
# 准备特征 X 和目标 y（与 Day 3 同一套，方便对比）
# 特征列：可能影响营收的输入
feature_cols = [
    'price', 'promotion', 'is_weekend', 'temp',
    'quality', 'competitors', 'holiday', 'location',
]
# 天气文字 → 数字分（晴最好，大雨最差）
weather_score_map = {'晴': 1.0, '多云': 0.8, '阴': 0.6, '小雨': 0.4, '大雨': 0.3}
df = df.copy()
df['weather_score'] = df['weather_label'].map(weather_score_map)

X = df[feature_cols + ['weather_score']].copy()  # 特征表
y = df['sales'].copy()  # 目标：营收

print('特征列:', list(X.columns))
print('样本数:', len(X))
X.head()


In [ ]:
from sklearn.model_selection import train_test_split

# test_size=0.2：约 20% 当测试集；random_state=42：随机种子，结果可复现
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('训练集行数:', len(X_train))
print('测试集行数:', len(X_test))


## 1. 对照表

| 类型 | 白话 | 例子 |
| --- | --- | --- |
| **参数** | 模型在 `fit` 时自己学出来的 | 树的切分、叶子均值；线性回归的系数 |
| **超参数** | 你先指定、学之前就要定的旋钮 | `n_estimators`、`max_depth`、`min_samples_leaf` |

人话：

> 超参数像考试前规定「用几支笔、卷子几页」；参数像答卷过程中写出的内容。

### 随机森林常见超参数（今天会碰）

| 旋钮 | 白话 | 调大时 |
| --- | --- | --- |
| `n_estimators` | 树的棵数 | 通常更稳，但更慢 |
| `max_depth` | 每棵树最多长多深 | 更贴训练，过拟合风险升 |
| `min_samples_leaf` | 叶子最少多少样本 | 越大越保守，越不容易背题 |


## 2. 动手：只改一个旋钮，看测试 R² 怎么变


In [ ]:
from sklearn.ensemble import RandomForestRegressor

for depth in [3, 5, 8, 12]:
    rf = RandomForestRegressor(
        n_estimators=100,
        max_depth=depth,
        random_state=42,
        n_jobs=-1,
    )
    rf.fit(X_train, y_train)
    tr = rf.score(X_train, y_train)
    te = rf.score(X_test, y_test)
    print(f'max_depth={depth:2d}  训练R²={tr:.3f}  测试R²={te:.3f}')


## 3. 观察提示

- 训练分随深度升高，很常见
- 测试分不一定一直涨——涨到某点后可能掉，就是过拟合苗头
- **只看训练分会选错旋钮** → 所以后面要用交叉验证 / 网格搜索

## 4. 小练习

1. `n_estimators` 和叶子上的「平均营收」哪个是超参数？
2. 为什么不能只在测试集上把所有旋钮试到最高分再交卷？


In [ ]:
# 在这里写
